# Autokernel Experiment Analysis

Analysis of autonomous kernel optimization results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 6 columns)
df = pd.read_csv("results.tsv", sep="\t")
df["speedup"] = pd.to_numeric(df["speedup"], errors="coerce")
df["runtime_us"] = pd.to_numeric(df["runtime_us"], errors="coerce")
df["ref_runtime_us"] = pd.to_numeric(df["ref_runtime_us"], errors="coerce")
df["status"] = df["status"].str.strip().str.lower()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("keep", 0)
n_discard = counts.get("discard", 0)
n_crash = counts.get("crash", 0) + counts.get("compile_error", 0)
n_incorrect = counts.get("incorrect", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")
print(f"Correctness rate: {n_keep + n_discard}/{len(df)} = {(n_keep + n_discard) / len(df):.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "keep"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    sp = row["speedup"]
    rt = row["runtime_us"]
    desc = row["description"]
    print(f"  #{i:3d}  speedup={sp:.4f}x  runtime={rt:.0f}us  {desc}")

## Speedup Over Time

Track how the best (kept) speedup evolves as experiments progress. The running maximum shows the frontier — the best speedup achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# All correct experiments (keep + discard)
correct = df[df["status"].isin(["keep", "discard"])].copy()
correct = correct.reset_index(drop=True)

# Plot discarded as faint background dots
disc = correct[correct["status"] == "discard"]
ax.scatter(disc.index, disc["speedup"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots
kept_v = correct[correct["status"] == "keep"]
ax.scatter(kept_v.index, kept_v["speedup"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running maximum step line
kept_mask = correct["status"] == "keep"
kept_idx = correct.index[kept_mask]
kept_sp = correct.loc[kept_mask, "speedup"]
running_max = kept_sp.cummax()
ax.step(kept_idx, running_max, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Reference line at 1.0x (PyTorch baseline)
ax.axhline(y=1.0, color="#e74c3c", linestyle="--", alpha=0.5, label="PyTorch baseline (1.0x)")

# Label each kept experiment
for idx, sp in zip(kept_idx, kept_sp):
    desc = str(correct.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (idx, sp),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

# Also plot failures as red x markers at y=0
failures = df[df["status"].isin(["crash", "compile_error", "incorrect"])].copy()
if len(failures) > 0:
    fail_idx = range(len(df))
    fail_mask = df["status"].isin(["crash", "compile_error", "incorrect"])
    ax.scatter(df.index[fail_mask], [0] * fail_mask.sum(),
               c="#e74c3c", s=15, marker="x", alpha=0.3, zorder=1, label="Failed")

n_total = len(df)
n_kept = len(df[df["status"] == "keep"])
best = kept_sp.max() if len(kept_sp) > 0 else 0
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Speedup over PyTorch (higher is better)", fontsize=12)
ax.set_title(f"Autokernel Progress: {n_total} Experiments, {n_kept} Kept, Best {best:.2f}x", fontsize=14)
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
kept = df[df["status"] == "keep"].copy()
if len(kept) == 0:
    print("No kept experiments yet.")
else:
    baseline_sp = kept.iloc[0]["speedup"]
    best_sp = kept["speedup"].max()
    best_row = kept.loc[kept["speedup"].idxmax()]

    print(f"Baseline speedup:    {baseline_sp:.4f}x")
    print(f"Best speedup:        {best_sp:.4f}x")
    print(f"Best runtime:        {best_row['runtime_us']:.0f} us")
    print(f"Reference runtime:   {best_row['ref_runtime_us']:.0f} us")
    print(f"Total improvement:   {best_sp - baseline_sp:.4f}x ({(best_sp / baseline_sp - 1) * 100:.1f}% over baseline)")
    print(f"Best experiment:     {best_row['description']}")
    print(f"\nBeat PyTorch (>1.0x): {'YES' if best_sp > 1.0 else 'NO'}")
    print()

    # Cumulative improvements
    print("Cumulative improvements:")
    for i, (_, row) in enumerate(kept.iterrows()):
        desc = str(row["description"]).strip()
        print(f"  Experiment #{i:3d}: speedup={row['speedup']:.4f}x  {desc}")

## Top Hits (Kept Experiments by Improvement Delta)

In [ ]:
kept = df[df["status"] == "keep"].copy()
if len(kept) <= 1:
    print("Need at least 2 kept experiments to show deltas.")
else:
    kept["prev_speedup"] = kept["speedup"].shift(1)
    kept["delta"] = kept["speedup"] - kept["prev_speedup"]

    # Drop baseline (no delta)
    hits = kept.iloc[1:].copy()
    hits = hits.sort_values("delta", ascending=False)

    print(f"{'Rank':>4}  {'Delta':>8}  {'Speedup':>10}  Description")
    print("-" * 80)
    for rank, (_, row) in enumerate(hits.iterrows(), 1):
        print(f"{rank:4d}  {row['delta']:+.4f}x  {row['speedup']:.4f}x    {row['description']}")

    print(f"\n{'':>4}  {hits['delta'].sum():+.4f}x  {'':>10}  TOTAL improvement over baseline")